# 07 · Colab DL 검증

ML과 같은 28개 기준일 `t`에서 `t+1`~`t+7`을 예측한다. 먼저 8개 시계열·2개 기준일로 N-HiTS와 TCN의 GPU 시험을 실행한다. **전체 실행과 TFT 셀은 시험 결과 확인 후 별도로 실행한다.**

`00` clone 노트북으로 저장소를 My Drive에 준비하고, GPU 런타임을 선택한다. 원본 `data/raw/train.csv`와 ML 예측 CSV가 Drive에 있어야 한다.


## 환경 준비


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/retail-demand-forecasting
%pip install -q -r requirements-neural.txt
%pip install -q -e . --no-deps


## Library


In [ ]:
from pathlib import Path

import pandas as pd
import torch

from retail_forecast.comparison import compare_on_common_rows
from retail_forecast.neural_data import load_neural_panel, load_validation_template
from retail_forecast.neural_runner import run_neural_validation


## 설정

`SMOKE_*`는 실행 흐름 확인용이다. 작은 시험 지표는 전체 ML 성능과 비교하지 않는다. 모델 학습에는 첫 검증 기준일 전의 판매량만 사용한다. 기준일 `t`의 예측에는 `t`까지의 실제 판매 이력과 미리 아는 달력 변수만 쓴다.


In [ ]:
REPO = Path.cwd()
ML_RUN_NAME = 'full_history_l2_i600'
ML_MODEL = 'lightgbm'
TEMPLATE_PATH = REPO / 'outputs' / ML_RUN_NAME / f'validation_predictions_{ML_MODEL}.csv'
SMOKE_SERIES = 8
SMOKE_ORIGINS = 2
SMOKE_TRAIN_DAYS = 120
SMOKE_STEPS = 20
FULL_STEPS = 300
INPUT_SIZE = 56
RUN_FULL = False
RUN_TFT = False

assert torch.cuda.is_available(), 'Colab GPU 런타임을 선택하세요.'
assert TEMPLATE_PATH.is_file(), f'ML 검증 예측 파일을 찾지 못했습니다: {TEMPLATE_PATH}'
print('GPU:', torch.cuda.get_device_name(0))


## ML 검증 기준일과 일별 입력

ML 결과에서 정확히 28개의 연속 기준일과 1~7일 목표를 검사한다. 원본 판매 CSV에서 필요한 열만 읽는다. 결측 날짜는 0 판매로 간주하지 않고 `available_mask=0`으로 표시한다.


In [ ]:
template = load_validation_template(TEMPLATE_PATH)
smoke_keys = template[['store_nbr', 'family']].drop_duplicates().head(SMOKE_SERIES)
smoke_template = template.merge(smoke_keys, on=['store_nbr', 'family'])
smoke_template = smoke_template.loc[smoke_template['date'].isin(sorted(template['date'].unique())[:SMOKE_ORIGINS])]
smoke_panel, smoke_series = load_neural_panel(REPO / 'data/raw', smoke_template)
smoke_panel = smoke_panel.loc[smoke_panel['ds'].ge(template['date'].min() - pd.Timedelta(days=SMOKE_TRAIN_DAYS))]
print(f'검증 기준일: {template.date.nunique()}개, {template.date.min().date()} ~ {template.date.max().date()}')
print(f'시험 입력: {len(smoke_series)}개 시계열, {len(smoke_panel):,}행')


## 작은 GPU 시험 · N-HiTS → TCN

학습 진행률은 NeuralForecast, 기준일별 예측은 tqdm으로 표시된다. 저장 경로는 `outputs/neural_smoke`와 `models/neural_smoke`이다. 두 모델의 결과와 소요시간을 보고 전체 실행 여부를 결정한다.


In [ ]:
nhits_smoke = run_neural_validation(
    'NHITS', smoke_panel, smoke_series, smoke_template,
    REPO / 'outputs/neural_smoke/nhits', REPO / 'models/neural_smoke/nhits',
    input_size=INPUT_SIZE, max_steps=SMOKE_STEPS, training_history_days=SMOKE_TRAIN_DAYS,
)
nhits_smoke


In [ ]:
tcn_smoke = run_neural_validation(
    'TCN', smoke_panel, smoke_series, smoke_template,
    REPO / 'outputs/neural_smoke/tcn', REPO / 'models/neural_smoke/tcn',
    input_size=INPUT_SIZE, max_steps=SMOKE_STEPS, training_history_days=SMOKE_TRAIN_DAYS,
)
tcn_smoke


In [ ]:
smoke_table = pd.concat([nhits_smoke, tcn_smoke], ignore_index=True)
smoke_table.to_csv(REPO / 'outputs/neural_smoke/comparison_nhits_tcn.csv', index=False)
smoke_table


## 전체 입력 · 승인 후 실행

위 시험의 GPU 메모리·학습시간·출력 형식을 확인한 후 `RUN_FULL=True`로 바꿔 아래 셀을 실행한다. 전체 입력은 Parquet으로 저장한다. N-HiTS와 TCN을 각각 실행해 본 뒤 TFT 여부를 정한다. TFT는 별도 `RUN_TFT=True` 설정이 필요하다.


In [ ]:
assert RUN_FULL, '작은 GPU 시험 결과를 확인한 뒤 RUN_FULL=True로 변경하세요.'
full_panel, full_series = load_neural_panel(REPO / 'data/raw', template)
processed = REPO / 'data/processed'
processed.mkdir(parents=True, exist_ok=True)
full_panel.to_parquet(processed / 'neural_panel_full.parquet', index=False)
full_series.to_parquet(processed / 'neural_series_full.parquet', index=False)
print(f'전체 입력: {len(full_series)}개 시계열, {len(full_panel):,}행')


In [ ]:
nhits_full = run_neural_validation(
    'NHITS', full_panel, full_series, template,
    REPO / 'outputs/neural_full/nhits', REPO / 'models/neural_full/nhits',
    input_size=INPUT_SIZE, max_steps=FULL_STEPS,
)
nhits_full


In [ ]:
tcn_full = run_neural_validation(
    'TCN', full_panel, full_series, template,
    REPO / 'outputs/neural_full/tcn', REPO / 'models/neural_full/tcn',
    input_size=INPUT_SIZE, max_steps=FULL_STEPS,
)
tcn_full


In [ ]:
common_paths = {
    'LightGBM': TEMPLATE_PATH,
    'NHITS': REPO / 'outputs/neural_full/nhits/validation_predictions_nhits.parquet',
    'TCN': REPO / 'outputs/neural_full/tcn/validation_predictions_tcn.parquet',
}
full_table = compare_on_common_rows(common_paths)
full_table.to_csv(REPO / 'outputs/neural_full/comparison_ml_dl.csv', index=False)
full_table


## TFT · N-HiTS/TCN 결과 확인 후


In [ ]:
assert RUN_TFT, 'N-HiTS와 TCN 전체 결과를 확인한 뒤 RUN_TFT=True로 변경하세요.'
tft_full = run_neural_validation(
    'TFT', full_panel, full_series, template,
    REPO / 'outputs/neural_full/tft', REPO / 'models/neural_full/tft',
    input_size=INPUT_SIZE, max_steps=FULL_STEPS,
)
tft_full


In [ ]:
common_paths['TFT'] = REPO / 'outputs/neural_full/tft/validation_predictions_tft.parquet'
full_table = compare_on_common_rows(common_paths)
full_table.to_csv(REPO / 'outputs/neural_full/comparison_ml_dl.csv', index=False)
full_table
